# <font style="font-family:roboto;color:#455e6c"> Generating datasets for Machine Learning Interatomic Potentials with the ASSYST method </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 16 March 2025 </font> </br> </br>
Marvin Poul, Sarath Menon, Haitham Gaafer, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

### References

- [Poul, M., Huber, L. & Neugebauer, J. Automated Generation of Structure Datasets for Machine Learning Potentials and Alloys. Preprint](https://doi.org/10.21203/rs.3.rs-4732459/v1) (2024)

In [1]:
from pyiron_core import Workflow, PyironFlow, as_function_node, as_inp_dataclass_node
from pyiron_core.pyiron_nodes.atomistic.assyst.stoichiometry import ElementInput, Stoichiometry, StoichiometryTable

## <font style="font-family:roboto;color:#455e6c"> Background </font> 

*Automated Small SYmetric Structure Training* or ASSYST is a method to generate training data for machine learning potentials.
The key idea is to use small structures to automatically explore structurally and chemically diverse atomic environments and provide training data around the energetically most favorable ones.

### <font style="font-family:roboto;color:#455e6c"> Workflow Overview </font>

![image](img/AssystSchematic.svg)

### <font style="font-family:roboto;color:#455e6c"> Transferability </font>

ASSYST trained potentials describe also structures that they are not directly trained on, such as point and planar defects.

![image](img/Fig8_MTP24_2d0_8d2_DefectsManual.png)

Liquid state is also well described and potentials are stable for long running thermodynamic integrations.

![image](img/Fig11_MgCa.png)

This phase diagram is our goal for today!

### <font style="font-family:roboto;color:#455e6c"> Literature </font>

- Mg and Defects: https://journals.aps.org/prb/abstract/10.1103/PhysRevB.107.104103
- Ternary Mg/Al/Ca: https://www.researchsquare.com/article/rs-4732459/v1

## <font style="font-family:roboto;color:#455e6c"> Constructing element combinations </font> 

The first step in the ASSYST workflow is to decide which chemical space to cover and how densely.
Increasing the new number of total atoms allows you to generate more and more complex structures
and also sample the chemical space more densely.

Here's an example for a ternary system, where we sampled the unaries with ASSYST datasets of 1-10 Atoms and the binaries and ternaries with 2-8 or 3-8 Atoms, respectively.

![img](img/Fig3_Everything_Conc_Plot.png)

Log-histogram of composition of a final training set.

`Elements` wraps a list of compositions at which we will sample random crystals.

In [2]:
mg = Stoichiometry((
    {'Mg': 2}, {'Mg': 4}
))
mg

Stoichiometry(stoichiometry=({'Mg': 2}, {'Mg': 4}))

In [3]:
al = Stoichiometry((
    {'Al': 1}, {'Al': 2}
))
al

Stoichiometry(stoichiometry=({'Al': 1}, {'Al': 2}))

Can be combined with standard python operations.

In [4]:
mg + al

Stoichiometry(stoichiometry=({'Mg': 2}, {'Mg': 4}, {'Al': 1}, {'Al': 2}))

In [5]:
mg | al

Stoichiometry(stoichiometry=({'Mg': 2, 'Al': 1}, {'Mg': 4, 'Al': 2}))

In [6]:
mg * al

Stoichiometry(stoichiometry=({'Mg': 2, 'Al': 1}, {'Mg': 2, 'Al': 2}, {'Mg': 4, 'Al': 1}, {'Mg': 4, 'Al': 2}))

Created by the ElementInput node and visualized by StoichiometryTable.

In [7]:
wf = Workflow("ASSYST_Elements_Unary")
wf.Element = ElementInput(element="Mg")
wf.ElementsTable = StoichiometryTable(wf.Element)

In [8]:
pf = PyironFlow([wf])
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

Build a small workflow that creates a table with Mg, Al and Ca so that:
1. Mg:Ca is always 2:1
2. combines it with 2-8 Al
3. has at least 2 Ca in every composition
4. contains at most 16 Atoms

Check `utilities` for nodes to `Add()`, `Multiply()` or `Or()` objects together.

Check `atomistic` -> `mlips` -> `fitting` -> `assyst` for nodes to `FilterSize()` or `ElementsTable()`
</div>

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Solution</b></p>
    Load this workflow for the solution
</div>

In [9]:
from pyiron_core.pyiron_nodes.basic.logic import Or
from pyiron_core.pyiron_nodes.basic.math import Multiply
from pyiron_core.pyiron_nodes.atomistic.assyst.stoichiometry import FilterSize

In [10]:
wf = Workflow("ASSYST_Elements_Combine")
wf.Mg = ElementInput(element="Mg", min_ion=4, max_ion=10, step_ion=2)
wf.Ca = ElementInput(element="Ca", min_ion=2, max_ion=10, step_ion=2)
wf.Al = ElementInput(element="Al", min_ion=1, max_ion=10, step_ion=1)
wf.combo = Or(x=wf.Mg, y=wf.Ca)
wf.product = Multiply(x=wf.combo, y=wf.Al)
wf.filter = FilterSize(elements=wf.product, min=0, max=16)
wf.ElementsTable = StoichiometryTable(wf.filter)

In [11]:
pf = PyironFlow([wf])
pf.gui

## <font style="font-family:roboto;color:#455e6c"> Full Workflow for a Small Structure Set </font> 

This demonstration uses the GRACE universal force fields for the relaxation steps.
Usually we would run them in low convergence DFT.

In [ ]:
from pyiron_core.pyiron_nodes.atomistic.assyst.plot import PlotSPG
from pyiron_core.pyiron_nodes.atomistic.assyst.stoichiometry import SpaceGroupSampling
from pyiron_core.pyiron_nodes.atomistic.calculator.optimize import RelaxLoop, GenericOptimizerSettings
from pyiron_core.pyiron_nodes.atomistic.structure.transform import RattleLoop, StretchLoop
from pyiron_core.pyiron_nodes.atomistic.engine.grace import Grace
from pyiron_core.pyiron_nodes.atomistic.structure.group import CombineStructures, SaveStructures

In [13]:
def make_assyst(
        name, *elements,
        min_atoms=2, max_atoms=4, max_structures=50,
        delete_existing_savefiles=False
):
    wf = Workflow(name)

    element_nodes = []
    if len(elements) > 0:
        e1, *elements = elements
        stoi = ElementInput(e1, min_atoms=min_atoms, max_atoms=max_atoms)
        setattr(wf, 'Element_1', stoi)
        element_nodes.append(stoi)
        for i, e in enumerate(elements):
            en = ElementInput(e, min_atoms=min_atoms, max_atoms=max_atoms)
            setattr(wf, f'Element_{i+2}', en)
            element_nodes.append(en)
            stoi = Multiply(stoi, en)
            setattr(wf, f'Multiply_{i+1}_{i+2}', stoi)
        if len(elements) > 0:
            wf.Multiply = stoi
        spg = SpaceGroupSampling(
                elements=stoi,
                spacegroups=None,
                max_atoms=len(element_nodes) * max_atoms,
                max_structures=max_structures
        )
    else:
        spg = SpaceGroupSampling(
                max_atoms=max_atoms,
                max_structures=max_structures
        )
    plotspg = PlotSPG(spg)

    calc_config = Grace()
    optimizer_settings = GenericOptimizerSettings()

    volume_relax = RelaxLoop(mode="volume", calculator=calc_config,
                             opt=optimizer_settings, structures=spg)
    full_relax = RelaxLoop(mode="full", calculator=calc_config,
                           opt=optimizer_settings,
                           structures=volume_relax.outputs.relaxed_structures)

    rattle = RattleLoop(
            structures=full_relax.outputs.relaxed_structures,
            sigma=0.25,
            samples=4
    )

    stretch = StretchLoop(
            structures=full_relax.outputs.relaxed_structures,
            hydro=0.8,
            shear=0.2,
            samples=4
    )

    combine_structures = CombineStructures(
            spg,
            volume_relax.outputs.relaxed_structures,
            full_relax.outputs.relaxed_structures,
            rattle,
            stretch
    )

    savestructures = SaveStructures(combine_structures, "data/Structures_Everything")

    wf.SpaceGroupSampling = spg
    wf.PlotSPG = plotspg

    wf.Calculator = calc_config
    wf.OptimizerSettings = optimizer_settings
    wf.VolumeRelax = volume_relax
    wf.FullRelax = full_relax

    wf.Rattle = rattle
    wf.Stretch = stretch

    wf.CombineStructures = combine_structures
    wf.SaveStructures = savestructures
    return wf

In [14]:
wf = make_assyst('ASSYST', 'Mg', 'Ca', 'Al')

wrapped:  <function Grace at 0x7fd2c4f477e0>


In [15]:
pf = PyironFlow([wf])

implement connect to self
implement connect to self
implement connect to self
implement connect to self
connected to node (self)
connected to node (self)
connected to node (self)
connected to node (self)


In [16]:
pf.gui

## <font style="font-family:roboto;color:#455e6c"> Precomputed Full Workflow with Large Structure Set </font> 

This is the same workflow, but pre-run with realistic input for a Unary system.
It contains ~10k structures and you can attach plotting functions at various nodes to view them.

In [17]:
wf = make_assyst('ASSYST_Mg_FULL', 'Mg', min_atoms=1, max_atoms=10, max_structures=None)

wrapped:  <function Grace at 0x7fd2c4f477e0>


In [18]:
pf = PyironFlow([wf])

implement connect to self
implement connect to self
implement connect to self
implement connect to self
connected to node (self)
connected to node (self)
connected to node (self)
connected to node (self)


In [19]:
pf.gui

<img src="img/logo_roll.png" width="1200">